# Live Client Output Demo

This notebook loads actual Alpaca and DeepSeek credentials from `fresh_simple_trading_project/.env` and calls the real clients with demo input variables defined in the notebook.

Clients exercised:

- `WebSearchNewsClient`
- `AlpacaNewsSearchClient`
- `AlpacaMarketDataClient`
- `AlpacaHistoricalMarketDataClient`
- `AlpacaAccountClient`
- `AlpacaBrokerClient`
- `DeepSeekLLMClient`
- `CombinedNewsSearchClient`

Notes:

- `AlpacaBrokerClient.place_order(...)` submits a real Alpaca paper order, so this notebook requires `PAPER=true`.
- If the Alpaca account does not include recent SIP market-data access, the market-data calls will print the real `APIError` response instead of bar data.
- Each call result is captured in `captured_outputs` and printed in the final cell.
- Use the project's `venv` Jupyter kernel so dependencies like `pandas`, `alpaca-py`, and `openai` are available.


In [12]:
import requests

url = "https://query1.finance.yahoo.com/v7/finance/quote?symbols=AAPL,TSLA&fields=regularMarketPrice,regularMarketTime&crumb=IF2ASvKzo4j"

try:
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    response.raise_for_status()
    data = response.json()
    print(data)
except requests.exceptions.RequestException as e:
    print("Request failed:", e)

Request failed: 401 Client Error: Unauthorized for url: https://query1.finance.yahoo.com/v7/finance/quote?symbols=AAPL,TSLA&fields=regularMarketPrice,regularMarketTime&crumb=IF2ASvKzo4j


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "fresh_simple_trading_project").exists():
            return candidate
        if (candidate / "fresh_simple_trading_project" / "src" / "fresh_simple_trading_project").exists():
            return candidate / "fresh_simple_trading_project"
    raise RuntimeError("Could not find the fresh_simple_trading_project root from the current working directory.")


project_root = find_project_root()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
load_dotenv(project_root / ".env")
project_root


PosixPath('/Users/madushika/IdeaProjects/group work/Agentic AI Workflow for Automated Trading/fresh_simple_trading_project')

In [2]:
from dataclasses import asdict, is_dataclass
from pprint import pprint

import pandas as pd

from fresh_simple_trading_project.alpaca_integration import (
    AlpacaAccountClient,
    AlpacaBrokerClient,
    AlpacaHistoricalMarketDataClient,
    AlpacaMarketDataClient,
    AlpacaService,
)
from fresh_simple_trading_project.config import Settings
from fresh_simple_trading_project.information_retrieval import (
    AlpacaNewsSearchClient,
    CombinedNewsSearchClient,
    WebSearchNewsClient,
)
from fresh_simple_trading_project.llm import DeepSeekLLMClient


In [3]:
settings = Settings.from_env(project_root=project_root)
settings.alpaca.require()
settings.llm.require()

if not settings.alpaca.paper_trading:
    raise RuntimeError(
        "This notebook expects PAPER=true before calling AlpacaBrokerClient.place_order."
    )

TEST_SYMBOL = "AAPL"
NEWS_QUERY = f"{TEST_SYMBOL} stock market news"
LLM_SYSTEM_PROMPT = "You are a concise trading analyst."
LLM_USER_PROMPT = "\n".join(
    [
        f"Symbol: {TEST_SYMBOL}",
        "Trend: bullish",
        "Signals: higher highs, stronger hourly momentum, healthy volume.",
        "News: AI growth narrative, broad market support, and manageable risk.",
        "Return one short trading note.",
    ]
)
BROKER_ORDER_QTY = 1
BROKER_ORDER_SIDE = "BUY"

captured_outputs: dict[str, object] = {}


def capture_call(name: str, fn, *args, **kwargs):
    try:
        result = fn(*args, **kwargs)
    except Exception as exc:
        result = f"{type(exc).__name__}: {exc}"
    captured_outputs[name] = result
    return result


def print_output(value: object) -> None:
    if isinstance(value, pd.DataFrame):
        print(f"rows={len(value)}, columns={list(value.columns)}")
        preview = value.tail(min(10, len(value)))
        print(preview.to_string())
        return
    if is_dataclass(value):
        pprint(asdict(value))
        return
    if isinstance(value, list) and value and all(is_dataclass(item) for item in value):
        pprint([asdict(item) for item in value])
        return
    pprint(value)


{
    "paper_trading": settings.alpaca.paper_trading,
    "trade_api_url": settings.alpaca.trade_api_url,
    "alpaca_enabled": settings.alpaca.enabled,
    "llm_model": settings.llm.model,
    "symbol": TEST_SYMBOL,
    "news_query": NEWS_QUERY,
    "broker_order": {"side": BROKER_ORDER_SIDE, "qty": BROKER_ORDER_QTY},
}


{'paper_trading': True,
 'trade_api_url': 'https://paper-api.alpaca.markets',
 'alpaca_enabled': True,
 'llm_model': 'deepseek-reasoner',
 'symbol': 'AAPL',
 'news_query': 'AAPL stock market news',
 'broker_order': {'side': 'BUY', 'qty': 1}}

In [4]:
service = AlpacaService(settings.alpaca)

alpaca_market_data_client = AlpacaMarketDataClient(
    service=service,
    five_minute_lookback_days=2,
    hourly_lookback_days=7,
)
alpaca_historical_market_data_client = AlpacaHistoricalMarketDataClient(
    service=service,
    history_lookback_days=30,
)
alpaca_account_client = AlpacaAccountClient(service=service)
alpaca_broker_client = AlpacaBrokerClient(service=service)
alpaca_news_client = AlpacaNewsSearchClient(service=service, max_age_days=settings.news.max_age_days)
web_news_client = WebSearchNewsClient(max_age_days=settings.news.max_age_days)
combined_news_client = CombinedNewsSearchClient(clients=[alpaca_news_client, web_news_client])
deepseek_client = DeepSeekLLMClient(settings.llm)


In [5]:
service.fetch_five_minute_bars('AAPL', 1)

,open,high,low,close,volume
timestamp,,,,,
2026-03-30 16:55:00+00:00,247.2400,247.3600,247.1400,247.3200,168376.0
2026-03-30 17:00:00+00:00,247.3200,247.4500,246.9700,246.9901,170155.0
2026-03-30 17:05:00+00:00,247.0000,247.3299,246.9400,246.9700,138983.0
2026-03-30 17:10:00+00:00,246.9700,247.0500,246.7600,246.8600,148138.0
2026-03-30 17:15:00+00:00,246.8650,246.9000,246.4401,246.5800,245264.0
...,...,...,...,...,...
2026-03-31 16:15:00+00:00,249.4350,249.8000,249.4350,249.7100,219153.0
2026-03-31 16:20:00+00:00,249.7100,250.5800,249.6350,250.5800,620663.0
2026-03-31 16:25:00+00:00,250.5700,250.6900,250.1250,250.3400,355443.0


In [6]:
capture_call(
    "AlpacaMarketDataClient.fetch_five_minute_bars",
    alpaca_market_data_client.fetch_five_minute_bars,
    TEST_SYMBOL,
)
capture_call(
    "AlpacaMarketDataClient.fetch_hourly_bars",
    alpaca_market_data_client.fetch_hourly_bars,
    TEST_SYMBOL,
)
capture_call(
    "AlpacaHistoricalMarketDataClient.fetch_five_minute_bars",
    alpaca_historical_market_data_client.fetch_five_minute_bars,
    TEST_SYMBOL,
)
capture_call(
    "AlpacaHistoricalMarketDataClient.fetch_hourly_bars",
    alpaca_historical_market_data_client.fetch_hourly_bars,
    TEST_SYMBOL,
)
capture_call(
    "AlpacaAccountClient.get_account_state",
    alpaca_account_client.get_account_state,
    TEST_SYMBOL,
)
capture_call(
    "AlpacaBrokerClient.place_order",
    alpaca_broker_client.place_order,
    TEST_SYMBOL,
    BROKER_ORDER_QTY,
    BROKER_ORDER_SIDE,
)
capture_call(
    "WebSearchNewsClient.search_news",
    web_news_client.search_news,
    NEWS_QUERY,
    limit=5,
)
capture_call(
    "AlpacaNewsSearchClient.search_news",
    alpaca_news_client.search_news,
    NEWS_QUERY,
    limit=5,
)
capture_call(
    "CombinedNewsSearchClient.search_news",
    combined_news_client.search_news,
    NEWS_QUERY,
    limit=5,
)
capture_call(
    "DeepSeekLLMClient.generate",
    deepseek_client.generate,
    LLM_SYSTEM_PROMPT,
    LLM_USER_PROMPT,
)

list(captured_outputs)


LLM generate for model 'deepseek-reasoner' is still thinking after 5.0s (waiting for streamed response).
LLM generate for model 'deepseek-reasoner' is still thinking after 10.0s (waiting for streamed response).
LLM generate for model 'deepseek-reasoner' completed after 13.2s.


['AlpacaMarketDataClient.fetch_five_minute_bars',
 'AlpacaMarketDataClient.fetch_hourly_bars',
 'AlpacaHistoricalMarketDataClient.fetch_five_minute_bars',
 'AlpacaHistoricalMarketDataClient.fetch_hourly_bars',
 'AlpacaAccountClient.get_account_state',
 'AlpacaBrokerClient.place_order',
 'WebSearchNewsClient.search_news',
 'AlpacaNewsSearchClient.search_news',
 'CombinedNewsSearchClient.search_news',
 'DeepSeekLLMClient.generate']

In [7]:
for name, value in captured_outputs.items():
    print(f"\n=== {name} ===")
    print_output(value)



=== AlpacaMarketDataClient.fetch_five_minute_bars ===
rows=275, columns=['open', 'high', 'low', 'close', 'volume']
                               open     high      low   close    volume
timestamp                                                              
2026-03-31 15:50:00+00:00  248.6200  249.084  248.575  248.91  258064.0
2026-03-31 15:55:00+00:00  248.9200  249.000  248.590  248.61  176409.0
2026-03-31 16:00:00+00:00  248.6100  249.300  248.490  249.04  443586.0
2026-03-31 16:05:00+00:00  249.0500  249.660  249.050  249.59  308350.0
2026-03-31 16:10:00+00:00  249.5850  249.690  249.180  249.44  206947.0
2026-03-31 16:15:00+00:00  249.4350  249.800  249.435  249.71  219153.0
2026-03-31 16:20:00+00:00  249.7100  250.580  249.635  250.58  620663.0
2026-03-31 16:25:00+00:00  250.5700  250.690  250.125  250.34  355443.0
2026-03-31 16:30:00+00:00  250.3499  250.500  250.180  250.41  212931.0
2026-03-31 16:35:00+00:00  250.3900  251.740  250.300  251.74  446520.0

=== AlpacaMarketDat

In [8]:
# from datetime import datetime
# from zoneinfo import ZoneInfo

# utc_str =  "2026-03-31T15:10:00Z"

# dt_utc = datetime.fromisoformat(utc_str.replace("Z", "+00:00"))
# dt_sri_lanka = dt_utc.astimezone(ZoneInfo("Asia/Colombo"))

# print(dt_sri_lanka.strftime("%Y-%m-%d %I:%M:%S %p"))

In [9]:
from fresh_simple_trading_project.alpaca_integration import (
    AlpacaService
)

In [10]:
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from functools import cached_property
import pandas as pd


def _normalize_ohlcv_bars(frame: pd.DataFrame) -> pd.DataFrame:
    required_columns = ["open", "high", "low", "close", "volume"]
    normalized = frame.copy().sort_index()
    if normalized.index.tz is None:
        normalized.index = pd.to_datetime(normalized.index, utc=True)
    else:
        normalized.index = pd.to_datetime(normalized.index, utc=True)
    normalized = normalized[~normalized.index.duplicated(keep="last")]
    missing = [column for column in required_columns if column not in normalized.columns]
    if missing:
        raise ValueError(f"OHLCV bars are missing columns: {missing}")
    normalized = normalized[required_columns].apply(pd.to_numeric, errors="coerce")
    normalized = normalized.dropna(subset=["open", "high", "low", "close"])
    normalized["volume"] = normalized["volume"].fillna(0.0)
    return normalized

def _normalize_alpaca_bars(frame: pd.DataFrame, symbol: str) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=["open", "high", "low", "close", "volume"])

    normalized = frame.copy()
    if isinstance(normalized.index, pd.MultiIndex) and "symbol" in normalized.index.names:
        normalized = normalized.xs(symbol, level="symbol", drop_level=True)

    if "symbol" in normalized.columns:
        normalized = normalized[normalized["symbol"] == symbol]

    if "timestamp" not in normalized.columns:
        normalized = normalized.reset_index()
    if "timestamp" not in normalized.columns:
        fallback_column = next(
            (column for column in normalized.columns if str(column).lower() in {"index", "level_0"}),
            normalized.columns[0],
        )
        normalized = normalized.rename(columns={fallback_column: "timestamp"})

    normalized["timestamp"] = pd.to_datetime(normalized["timestamp"], utc=True)
    normalized = normalized.sort_values("timestamp").drop_duplicates(subset=["timestamp"]).set_index("timestamp")
    return _normalize_ohlcv_bars(normalized)


@dataclass
class AlpacaService2(AlpacaService):  
    config: AlpacaConfig

    def stock_data_client(self) -> Any:
        self.config.require()
        from alpaca.data.historical.stock import StockHistoricalDataClient

        return StockHistoricalDataClient(
            api_key=self.config.api_key,
            secret_key=self.config.api_secret,
        )

    @cached_property
    def stock_data_client(self) -> Any:
            self.config.require()
            from alpaca.data.historical.stock import StockHistoricalDataClient

            return StockHistoricalDataClient(
                api_key=self.config.api_key,
                secret_key=self.config.api_secret,
            )

    def fetch_five_minute_bars(self, symbol: str, lookback_days: int=None) -> pd.DataFrame:
            from alpaca.data.requests import StockBarsRequest
            from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

            end = datetime.now(timezone.utc)
            request = StockBarsRequest(
                symbol_or_symbols=[symbol],
                timeframe=TimeFrame(amount=5, unit=TimeFrameUnit.Minute),
                start=end - timedelta(days=lookback_days) if lookback_days else None,
                # end=end,
            )
            frame = self.stock_data_client.get_stock_bars(request).df
            return _normalize_alpaca_bars(frame, symbol)

In [11]:

a = AlpacaService2(settings.alpaca)

# a.fetch_five_minute_bars("AAPL")

a.fetch_hourly_bars("AAPL")

,open,high,low,close,volume
timestamp,,,,,
2026-03-31 08:00:00+00:00,248.05,248.6500,247.410,247.9600,66563.0
2026-03-31 09:00:00+00:00,248.01,248.4400,247.980,248.4000,12402.0
2026-03-31 10:00:00+00:00,248.45,248.7500,248.000,248.0800,22906.0
2026-03-31 11:00:00+00:00,248.30,249.0000,247.500,248.8900,186401.0
2026-03-31 12:00:00+00:00,248.98,249.5058,248.300,248.3000,202362.0
2026-03-31 13:00:00+00:00,248.18,249.2700,247.320,248.7927,3651706.0
2026-03-31 14:00:00+00:00,248.82,249.1200,247.101,247.3200,3774428.0
2026-03-31 15:00:00+00:00,247.29,249.0840,247.270,248.6100,2911440.0
2026-03-31 16:00:00+00:00,248.61,253.3500,248.490,252.8500,5003097.0
